In [2]:
import pandas as pd

df = pd.read_csv("source.csv")
df = df.drop(df.index[-1:])

In [3]:

df["delta"] = (df["close"] - df["open"]) / df["open"]

In [4]:
import numpy as np

statLength = 100

def _rolling_zscore_last(x):
    std = np.std(x)
    if std == 0 or np.isnan(std):
        return np.nan
    return (x[-1] - np.mean(x)) / std

df.loc[:, "price_effect"] = (df["high"] - df["low"]) * 0.25 + abs(df["close"] - df["open"]) * 0.75
df.loc[:, "vol_min"] = df["volume"].rolling(window=statLength, min_periods=statLength).min()
# avoid division by zero
df.loc[:, "vol_min"] = df.loc[:, "vol_min"].replace(0, np.nan)
df.loc[:, "vol_norm"] = df["volume"] / df.loc[:, "vol_min"]
df.loc[:, "pricePerVol"] = df.loc[:, "price_effect"] / df.loc[:, "vol_norm"]
# compute z-score within a rolling window and keep the z-score of the last element in each window
df.loc[:, "sentiment_zs"] = df["pricePerVol"].rolling(window=statLength, min_periods=statLength).apply(lambda x: _rolling_zscore_last(x), raw=True)


In [5]:

col = ["close", "volume", "delta", "sentiment_zs", "takerbuyvolume"]
df.to_csv("target.csv", columns=col, index=False)
df.tail()

,opentime,open,high,low,close,volume,closetime,quoteassetvolume,numberoftrades,takerbuyvolume,takerbuyquotevolume,ignore,delta,price_effect,vol_min,vol_norm,pricePerVol,sentiment_zs
56883,1772740800000,70875.5,71360.9,70805.9,71206.2,5278.268,1772744399999,3.753893e+08,212931,2544.988,1.809924e+08,0,0.004666,386.775,3720.637,1.418646,272.636663,2.672603
56884,1772744400000,71205.8,71530.4,70974.5,71099.7,5394.081,1772747999999,3.845030e+08,143669,2673.149,1.905848e+08,0,-0.001490,218.550,3720.637,1.449774,150.747684,0.640595
56885,1772748000000,71099.6,71340.0,71022.1,71170.9,3506.972,1772751599999,2.496331e+08,98493,1865.237,1.327864e+08,0,0.001003,132.950,3506.972,1.000000,132.950000,0.340175
56886,1772751600000,71170.9,71268.1,70737.9,70854.8,4211.503,1772755199999,2.989753e+08,118972,2143.634,1.522149e+08,0,-0.004441,369.625,3506.972,1.200894,307.791429,3.063605
56887,1772755200000,70854.8,71024.9,70740.0,70952.5,3722.132,1772758799999,2.639054e+08,115501,2045.096,1.450013e+08,0,0.001379,144.500,3506.972,1.061352,136.147094,0.346945
